In [1]:
import plotly.graph_objects as go
import trimesh
import numpy as np
from campaign import *
from cir import *
from constants import *
from diffraction import *
from em_core import *
from engine_hybrid import *
from engine_image import *
from engine_image_parallel import *
from engine_sbr import *
from geometry import *
from legacy import *
from terrain_io import *
from timing import *
from viz import *
from scenarios import *
from orbit_helpers import *
import plotly.io as pio
import pickle
from polyscope_live import launch_live
from terrain_class import MultiPath
from pathlib import Path
pio.renderers.default = 'browser'

BASE_DIR = Path.cwd().resolve().parent
tifs_dir = BASE_DIR / "Multipath" / "tifs_new"
kernels_dir = BASE_DIR / "Multipath" / "kernels"
meshes_dir = BASE_DIR / "Multipath" / "meshes"

Class Loaded


In [ ]:
# path = r"/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/tifs_new/LDEM_875S_20M.tif"
# tt = MultiPath(dem_path=path, frame="local", launch_mode="surface", utc_time="2026-06-22T00:00:00", 
#                num_rays=1_000_000, polarization=True, isotropic=False, pol_convention="fixed", verbose=True)

# visualize_dem_2d(path, downsample_factor=5, title="Full DEM")
# tt.crop(r"/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/tifs_new/De_Garlache.tif",  method = "center_xy_km", center_x_km=-50.477, center_y_km = -3.252, side_km = 14)

# visualize_dem_2d(r"/Users/pierazhi/Desktop/Multipath/Multipath/Refactored/tifs_new/De_Garlache.tif", downsample_factor=5, title="Full DEM")


In [10]:
path = tifs_dir / "LDEM_875S_20M.tif"
utc0 = "2026-06-22T00:00:00"

scene = MultiPath(dem_path=path, frame="local", launch_mode="surface", utc_time=utc0,
               num_rays=1_000_000, polarization=True, isotropic=False, pol_convention="fixed", verbose=True)
scene.gen_mesh(target_resolution_m=220)
visualize_mesh_3D(scene.mesh_local, show_axes=True, show_edges=False)

mesh_pa = scene.mesh_pa
mesh_local = scene.mesh_local

Native DEM resolution:      20.000 m × 20.000 m
Requested mesh resolution:  220
Output DEM shape:           689 × 689
Effective mesh resolution:  220.145 m × 220.145 m
Resampling method:          lanczos
Vertices:                  474,721
Faces kept:                946,688


In [11]:
R_M = 1_737_400.0
MU_MOON = 4.9048695e12

alt = 100e3

R_a = (R_M + alt)*1
R_p = (R_M + alt)*1

a = (R_a + R_p) / 2
e = 1 - R_p / a
i = 90.0
Om = 0.0
om = -90
theta = 0.0   

kep0_pa = np.array([
    a,
    e,
    np.deg2rad(i),
    np.deg2rad(Om),
    np.deg2rad(om),
    np.deg2rad(theta)
])

r0_pa, v0_pa = kep2car(kep0_pa, MU_MOON)

et0 = spice.str2et(utc0)
R_pa2i_0 = spice.pxform("MOON_PA", "J2000", et0)

r0_i = R_pa2i_0 @ r0_pa
v0_i = R_pa2i_0 @ v0_pa

y0 = np.hstack([r0_i, v0_i])

T = 2*np.pi*np.sqrt(a**3 / MU_MOON)
n_orbits = 1
tspan = np.linspace(0, n_orbits*T, 10000)
time_step = tspan[1] - tspan[0]

print(f"Time Step: {time_step:.2f} s")    

sol = solve_ivp(
    tbp,
    [0.0, n_orbits*T],
    y0,
    t_eval=tspan,
    rtol=1e-10,
    atol=1e-12,
    args=(MU_MOON, 'no')
)

YY_i = sol.y.T

traj_pa = np.zeros((len(tspan), 3))
traj_local = np.zeros((len(tspan), 3))

for k, t in enumerate(tspan):
    et = et0 + t

    R_i2pa = spice.pxform("J2000", "MOON_PA", et)
    traj_pa[k] = R_i2pa @ YY_i[k, :3]

    traj_local[k] = pa_to_local(
        traj_pa[k],
        scene.origin_pa,
        scene.A_local_to_pa
    )

Time Step: 0.71 s


In [12]:
fig = go.Figure()

fig.add_trace(go.Mesh3d(
            x=mesh_local.vertices[:, 0], y=mesh_local.vertices[:, 1], z=mesh_local.vertices[:, 2],
            i=mesh_local.faces[:, 0], j=mesh_local.faces[:, 1], k=mesh_local.faces[:, 2],
            color="dimgray",
            showscale=False,
            opacity=0.5,
            flatshading=False,
            name="Mesh",
            lighting=dict(
                ambient=0.2,
                diffuse=0.8,
                specular=0.5,
                roughness=0.5,
                fresnel=0.2
            ),
            lightposition=dict(x=1, y=1, z=-5)
        ))

twindow = 25

fig.add_trace(go.Scatter3d(
        x=traj_local[:twindow, 0],
        y=traj_local[:twindow, 1],
        z=traj_local[:twindow, 2],
        mode='lines',
        name="Trajectory PA",
        line=dict(color='red', width=3) # Corretto qui
    ))

fig.add_trace(go.Scatter3d(
        x=traj_local[-twindow:,  0],
        y=traj_local[-twindow:,  1],
        z=traj_local[-twindow:,  2],
        mode='lines',
        name="Trajectory PA",
        line=dict(color='red', width=3) # Corretto qui
    ))

# fig.add_trace(go.Scatter3d(
#     x=traj_local[:twindow:5, 0],
#     y=traj_local[:twindow:5, 1],
#     z=traj_local[:twindow:5, 2],
#     mode='lines+markers',          # Mostra sia la linea che i punti distanziati
#     name="Trajectory PA (Inizio)",
#     line=dict(color='red', width=3),
#     marker=dict(size=4, color='darkred') # Opzionale: stile per i singoli punti
# ))

# # 2. Ultimi elementi da '-twindow' alla fine, prendendo un punto ogni 5
# fig.add_trace(go.Scatter3d(
#     x=traj_local[-twindow::5, 0],
#     y=traj_local[-twindow::5, 1],
#     z=traj_local[-twindow::5, 2],
#     mode='lines+markers',          # Mostra sia la linea che i punti distanziati
#     name="Trajectory PA (Fine)",
#     line=dict(color='red', width=3),
#     marker=dict(size=4, color='darkred') # Opzionale: stile per i singoli punti
# ))

# fig.add_trace(go.Scatter3d(
#         x=traj_local[-int(np.round(len(tspan)/2)) -twindow: -int(np.round(len(tspan)/2)) +twindow,  0],
#         y=traj_local[-int(np.round(len(tspan)/2)) -twindow: -int(np.round(len(tspan)/2)) +twindow, 1],
#         z=traj_local[-int(np.round(len(tspan)/2)) -twindow: -int(np.round(len(tspan)/2)) +twindow, 2],
#         mode='lines',
#         name="Trajectory PA",
#         line=dict(color='red', width=3) # Corretto qui
#     ))

fig.update_layout(
        title=f"Max Altitude: {(np.max(traj_local[:, 2])*1e-3):.2f} km | Total Traj Time: {(time_step * twindow * 2 / 60):.2f} min | e = {e:.2f}", 
        scene=dict(
            aspectmode="data",
            bgcolor="#f0f0f8",
        ),   
    )
fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Mesh3d(
            x=mesh_pa.vertices[:, 0], y=mesh_pa.vertices[:, 1], z=mesh_pa.vertices[:, 2],
            i=mesh_pa.faces[:, 0], j=mesh_pa.faces[:, 1], k=mesh_pa.faces[:, 2],
            color="dimgray",
            showscale=False,
            opacity=0.5,
            flatshading=False,
            name="Mesh",
            lighting=dict(
                ambient=0.2,
                diffuse=0.8,
                specular=0.5,
                roughness=0.5,
                fresnel=0.2
            ),
            lightposition=dict(x=1, y=1, z=-5)
        ))
fig.add_trace(go.Scatter3d(
        x=traj_pa[:, 0],
        y=traj_pa[:, 1],
        z=traj_pa[:, 2],
        mode='lines',
        name="Trajectory PA",
        line=dict(color='red', width=3) # Corretto qui
    ))

# fig.add_trace(go.Scatter3d(
#         x=YY_i[:, 0],
#         y=YY_i[:, 1],
#         z=YY_i[:, 2],
#         mode='lines',
#         name="Trajectory ECI",
#         line=dict(color='blue', width=3) # Corretto qui
#     ))

add_earth_surface(fig, R_M, opacity = 0.05)
fig.update_layout(
        title=f"e = {e:.2f}", 
    )
fig.show()


In [ ]:
for i in range(0, 50, 10):
    scene.add_tx(traj_local[i], n=1, spacing=20, mode="absolute", coords="local", bounded=False,
                polarization="RHCP", boresight_coords="local", boresight=np.array([0,0,-1]), pattern=None)
    scene.add_rx([0, 0], n=5, height=5, spacing=20, mode="terrain", coords="local",
                polarization="RHCP", boresight_coords="local", boresight=np.array([0,0,1]), pattern=None)
    scene.path_solver(len(scene.pos_tx) - 1, 0, plot=True, long_distance=False, detailed=True, show_edges=False)